# Eye Tracking benchmark: 4. Train the final deployment selector

This notebook produces one deployment selector after the nested evaluation has been reviewed. It is not a further performance evaluation: all eligible benchmark gaps are used for fitting, so notebook 3 remains the only valid source of held-out performance estimates.


## Final-model selection and training scope

One hyperparameter candidate is selected using its mean inner-fold selected nRMSE across the completed outer folds. The final shared multi-output random forest then learns every candidate-method error target from all eligible benchmark gaps. Missing feature values are median-imputed from final-training rows.

The final scale floor is estimated from all final-training gaps. This is appropriate for deployment and intentionally differs from the fold-local scale floors used in confirmatory evaluation.


In [1]:
from pathlib import Path
import os
import subprocess
import sys

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / 'pyproject.toml').is_file():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError('Start Jupyter from inside the repository.')
    PROJECT_ROOT = PROJECT_ROOT.parent

# Read ignored workstation paths without placing them in this notebook.
source_root = str(PROJECT_ROOT / 'src')
if source_root not in sys.path:
    sys.path.insert(0, source_root)
from gap_imputation_benchmark.paths import load_local_environment, local_path_or_default
# The local .env file is authoritative for this workstation notebook.
load_local_environment(override=True)

BENCHMARK_DIR = local_path_or_default(
    'EYE_TRACKING_BENCHMARK_DIR', PROJECT_ROOT / 'benchmarks' / 'eyetracking',
)
RESULTS_DIR = local_path_or_default(
    'EYE_TRACKING_LODO_RESULTS_DIR', PROJECT_ROOT / 'results' / 'eyetracking' / 'lodo',
)
ARTIFACT_DIR = local_path_or_default(
    'EYE_TRACKING_ARTIFACT_DIR', PROJECT_ROOT / 'artifacts' / 'eyetracking',
)
SCRIPT = PROJECT_ROOT / 'scripts' / 'train_final_native_7_selector.py'

command = [
    sys.executable, str(SCRIPT),
    '--benchmark-dir', str(BENCHMARK_DIR),
    '--artifact-dir', str(ARTIFACT_DIR),
]
# Run directly from src/ so the notebook works before an editable package install.
execution_env = os.environ.copy()
execution_env['PYTHONPATH'] = source_root + os.pathsep + execution_env.get('PYTHONPATH', '')
subprocess.run(command, cwd=PROJECT_ROOT, env=execution_env, check=True)
try:
    output_location = ARTIFACT_DIR.relative_to(PROJECT_ROOT).as_posix()
except ValueError:
    output_location = 'configured artifact directory'
print(f'Selector artifact outputs: {output_location}')

 candidate_index  n_estimators  min_samples_leaf  max_depth max_features  macro_lodo_selected_nrmse  held_out_GazeBase_selected_nrmse  held_out_GazeBaseVR_selected_nrmse  held_out_Pedrotti_selected_nrmse  held_out_ZuCo_selected_nrmse
               4           300                10        NaN          1.0                   0.221233                          0.178440                            0.164515                          0.381615                      0.160362
               7           500                10       12.0          0.7                   0.221318                          0.178980                            0.164709                          0.381700                      0.159884
               5           300                 5       12.0          0.7                   0.221334                          0.178940                            0.164383                          0.381897                      0.160115
               1           400                 5        NaN     

## Reproducible artifact creation

The command consumes the frozen configuration, completed nested-evaluation tuning table, and benchmark table (plus required external data where applicable). It does not overwrite an existing artifact directory. Use a new configured `EYE_TRACKING_ARTIFACT_DIR` path for a deliberate retraining run.

The artifact stores the fitted model and feature imputer; its `metadata.json` is human-readable and contains no machine-specific source-data path.


In [2]:
import json
import pandas as pd

metadata = json.loads((ARTIFACT_DIR / 'metadata.json').read_text(encoding='utf-8'))
training_scope = metadata.get('training_datasets', metadata.get('training_groups'))
artifact_overview = {
    'domain': metadata.get('domain'),
    'training_rows': metadata.get('training_rows'),
    'training_scope': training_scope,
    'feature_scale_floor': metadata.get('feature_scale_floor'),
}

# LODO performance is evaluation output, not final-artifact metadata.
summary_path = RESULTS_DIR / 'lodo_summary.csv'
if summary_path.is_file():
    lodo_summary = pd.read_csv(summary_path)
    aggregate = lodo_summary.loc[lodo_summary['evaluation_set'] == 'macro_mean_across_datasets']
    per_dataset = lodo_summary.loc[lodo_summary['evaluation_set'] != 'macro_mean_across_datasets']
    lodo_overview = {
        'macro_lodo_selected_nrmse': (aggregate['mean_selected_nrmse'].iloc[0] if not aggregate.empty else per_dataset['mean_selected_nrmse'].mean()),
        'per_dataset_selected_nrmse': per_dataset.set_index('evaluation_set')['mean_selected_nrmse'].to_dict(),
    }
else:
    lodo_overview = {'status': f'LODO summary not found: {summary_path}'}

{'artifact': artifact_overview, 'lodo_evaluation': lodo_overview}

{'artifact': {'domain': 'eye_tracking',
  'training_rows': 1599,
  'training_scope': ['GazeBase', 'GazeBaseVR', 'Pedrotti', 'ZuCo'],
  'feature_scale_floor': {'value': 0.012457855000000195,
   'source': 'first percentile of positive local-context IQRs across final training gaps',
   'test_data_contribute_to_floor': False}},
 'lodo_evaluation': {'macro_lodo_selected_nrmse': np.float64(0.2213090177292267),
  'per_dataset_selected_nrmse': {'GazeBase': 0.1791653280184401,
   'GazeBaseVR': 0.1642562291591712,
   'Pedrotti': 0.3816998752264464,
   'ZuCo': 0.160114638512849}}}

## Artifact review checklist

Before deployment, verify the domain, candidate-method order, feature order, selected hyperparameters and their nested-evaluation provenance, training-row count, training scope, and scale-floor value. These fields must agree with the reviewed benchmark and evaluation outputs.

The selector predicts method errors and chooses the lowest predicted error for a real missing interval. Reconstruction of the missing values remains the responsibility of the selected candidate imputation method.
